In [ ]:
from sentiment_classifier.utils.utils import get_root_dir
from sentiment_classifier.data.retrieval import read_csv

from sentiment_classifier.data.retrieval import get_train_test_data

In [ ]:
root = get_root_dir()
data_path = root / "data" /"all-data.csv"

In [ ]:
df = read_csv(data_path, encoding='latin-1', header=None, names = ["sentiment", "text"])

In [ ]:
train, test = get_train_test_data(df, test_size=0.2)

# Preparing the model

In [ ]:
import spacy
nlp=spacy.load("en_core_web_trf")

In [ ]:
#Creating tuples
train['tuples'] = train.apply(lambda row: (row['text'],row['sentiment']), axis=1)
train_tuple = train['tuples'].tolist()

test['tuples'] = test.apply(lambda row: (row['text'],row['sentiment']), axis=1)
test_tuple = test['tuples'].tolist()


In [ ]:
def get_list_of_tuples(df):
    """
    Get list of tuples from a dataframe.

    :param df: dataframe
    :return: list of tuples
    """
    df['tuples'] = df.apply(lambda row: (row['text'],row['sentiment']), axis=1)
    return df['tuples'].tolist()

In [ ]:
train_tuple

In [ ]:
for doc, label in nlp.pipe(train_tuple, as_tuples=True):
    print(doc, label)

In [ ]:
help(doc)

In [ ]:
def create_spacy_documents(data):
  """
  This function takes in a list of tuples and returns a list of spacy documents.
  """
  text = []
  for doc, label in nlp.pipe(data, as_tuples = True):
    if (label=='positive'):
      doc.cats['positive'] = 1
      doc.cats['negative'] = 0
      doc.cats['neutral']  = 0
    elif (label=='negative'):
      doc.cats['positive'] = 0
      doc.cats['negative'] = 1
      doc.cats['neutral']  = 0
    else:
      doc.cats['positive'] = 0
      doc.cats['negative'] = 0
      doc.cats['neutral']  = 1
      text.append(doc)
  return(text)


In [ ]:
type(nlp)

In [ ]:
isinstance(type(nlp), spacy.lang.en.English)

In [ ]:
help(nlp)

In [3]:
from sentiment_classifier.utils.utils import load_spacy_model
from sentiment_classifier.utils.utils import get_root_dir
from sentiment_classifier.data.retrieval import get_train_test_data
from sentiment_classifier.data.retrieval import read_csv
from sentiment_classifier.data.training_dataset import get_list_of_tuples
from sentiment_classifier.data.training_dataset import create_spacy_documents
import mlflow 
import spacy


In [5]:
nlp = load_spacy_model(model_path="models/model-best")

c:\Users\manue\Documents\NLP_usecases\sentiment_classifier\.venv\Lib\site-packages\transformers\utils\generic.py:441: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(
c:\Users\manue\Documents\NLP_usecases\sentiment_classifier\.venv\Lib\site-packages\transformers\utils\generic.py:309: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _torch_pytree._register_pytree_node(


In [6]:
example = "According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing"
doc = nlp(example)

In [7]:
doc.cats

{'positive': 0.45736005902290344,
 'negative': 0.043489351868629456,
 'neutral': 0.4991506040096283}

In [6]:
with mlflow.start_run(run_name = "logging_model") as run:
    mlflow.spacy.log_model(spacy_model=nlp, artifact_path="sentiment_classifier")

In [179]:
loaded_model = mlflow.pyfunc.load_model(model_uri=f"runs:/{run.info.run_id}/sentiment_classifier")
spacy_model = mlflow.spacy.load_model(model_uri=f"runs:/{run.info.run_id}/sentiment_classifier")

In [13]:
nlp = spacy.blank("en")

In [75]:

root = get_root_dir() 
data_path = root / "data" /"all-data.csv" 
df = read_csv(data_path, encoding='latin-1', header=None, names = ["sentiment", "text"])

train, test = get_train_test_data(df)
train_data = get_list_of_tuples(train)
test_data = get_list_of_tuples(test)

test_docs = create_spacy_documents(nlp = nlp, data = test_data)

In [76]:
test_docs[0].cats

{'positive': 0, 'negative': 0, 'neutral': 1}

In [77]:
test["text"]

3207    The company was supposed to deliver machinery ...
1684    UNC Charlotte would also deploy SSH Tectia Con...
1044    In 2009 , Lee & Man had a combined annual prod...
4145    `` That 's a very high figure on the European ...
1538    In Finland , the corresponding service is Alma...
                              ...                        
3691    News Corp. 's MySpace.com Web site will displa...
1507    Both Mr Walden and Mr Ignatius will be respons...
1126    `` Every partner will be allowed to buy a quan...
180     Diluted earnings per share ( EPS ) rose to EUR...
1215    Entire paper mills may be set up , especially ...
Name: text, Length: 970, dtype: object

In [ ]:
from mlflow.metrics import make_metric
from mlflow.metrics import MetricValue


def eval_fn(predictions, targets, context):
    values = predictions.iloc[0].values[0]
    
    scores = (predictions == targets) + context
    return MetricValue(
        scores=list(scores),
        aggregate_results={"mean": np.mean(scores), "sum": np.sum(scores)},
    )


mymetric = make_metric(eval_fn=eval_fn, greater_is_better=False, name="mymetric")


In [84]:
predictions = loaded_model.predict(test[["text"]])

In [89]:
predictions.iloc[0].values[0]

{'positive': 0.21659491956233978,
 'negative': 0.03428983315825462,
 'neutral': 0.7491152882575989}

In [134]:
from mlflow.metrics import EvaluationMetric
from mlflow.metrics import MetricValue
import pandas as pd
import operator
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import f1_score

In [152]:
def eval_fn(predictions: pd.Series, targets: pd.Series)->float:
    """
    """ 
    df = pd.DataFrame(predictions, columns=["predictions"])
    df["prediction"] = df["predictions"].apply( lambda x : max(x.items(), key = operator.itemgetter(1))[0])
    df["target"] = targets
    accuracy_score = balanced_accuracy_score(df["target"], df["prediction"])
    return accuracy_score
    


In [161]:
eval_fn(predictions, test["sentiment"])

0.7136766252995294

In [153]:
custom_acc = EvaluationMetric(eval_fn=eval_fn, name="custom_acc", greater_is_better=True)

In [175]:
loaded_model.predict(test[["text"]])

,predictions
3207,"{'positive': 0.21659491956233978, 'negative': ..."
1684,"{'positive': 0.34155046939849854, 'negative': ..."
1044,"{'positive': 0.8569769263267517, 'negative': 0..."
4145,"{'positive': 0.255377858877182, 'negative': 0...."
1538,"{'positive': 0.24798519909381866, 'negative': ..."
...,...
3691,"{'positive': 0.2780744135379791, 'negative': 0..."
1507,"{'positive': 0.17686673998832703, 'negative': ..."
1126,"{'positive': 0.16717375814914703, 'negative': ..."
180,"{'positive': 0.9693070650100708, 'negative': 0..."


In [176]:
labels = {"positive": 2, "negative": 0, "neutral": 1}
test["label"] = test["sentiment"].apply(lambda x: labels[x])

In [177]:
model_uri = f"runs:/{run.info.run_id}/sentiment_classifier"
result = mlflow.evaluate(
    model = model_uri,
    data=test[["text", "sentiment"]],
    model_type=None,
    targets = "sentiment",
    evaluators = None,
    extra_metrics = [custom_acc],
)

# result = mlflow.evaluate(
#     model=loaded_model,
#     data = eval_data,
#     model_type=None, # to avoid internal metrics calculation
#     targets="target",
#     extra_metrics=[custom_acc],
# )

c:\Users\manue\Documents\NLP_usecases\sentiment_classifier\.venv\Lib\site-packages\mlflow\data\digest_utils.py:26: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  string_columns = trimmed_df.columns[(df.applymap(type) == str).all(0)]
c:\Users\manue\Documents\NLP_usecases\sentiment_classifier\.venv\Lib\site-packages\mlflow\models\evaluation\base.py:521: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data = data.applymap(_hash_array_like_element_as_bytes)
c:\Users\manue\Documents\NLP_usecases\sentiment_classifier\.venv\Lib\site-packages\mlflow\models\evaluation\base.py:521: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data = data.applymap(_hash_array_like_element_as_bytes)
2024/04/14 10:06:31 INFO mlflow.models.evaluation.base: Evaluating the model with the default evaluator.
2024/04/14 10:06:31 INFO mlflow.models.evaluation.default_evaluator: Computing model predictions.
2024/

MlflowException: Metric 'custom_acc': Error:
ValueError("Classification metrics can't handle a mix of binary and unknown targets")
Traceback (most recent call last):
  File "c:\Users\manue\Documents\NLP_usecases\sentiment_classifier\.venv\Lib\site-packages\mlflow\models\evaluation\default_evaluator.py", line 1615, in _test_first_row
    metric_value = _evaluate_metric(metric_tuple, eval_fn_args)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\manue\Documents\NLP_usecases\sentiment_classifier\.venv\Lib\site-packages\mlflow\models\evaluation\default_evaluator.py", line 598, in _evaluate_metric
    metric = metric_tuple.function(*eval_fn_args)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\manue\AppData\Local\Temp\ipykernel_12728\165372255.py", line 7, in eval_fn
    accuracy_score = balanced_accuracy_score(df["target"], df["prediction"])
                     ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\manue

In [170]:
result

NameError: name 'result' is not defined

In [182]:
test_docs

[The company was supposed to deliver machinery to a veneer mill in the Tomsk region , in Russia .,
 UNC Charlotte would also deploy SSH Tectia Connector to enable secure application connectivity .,
 In 2009 , Lee & Man had a combined annual production capacity of close to 4.5 million tonnes of paper and 300,000 tonnes of pulp .,
 `` That 's a very high figure on the European scale , '' Noop said , recalling however that this also includes beer bought by Finnish tourists .,
 In Finland , the corresponding service is Alma Media 's Etuovi.com , Finland 's most popular and best known nationwide online service for home and property sales .,
 Construction is scheduled to start in April-June 2007 and to be completed in early 2008 .,
 Finnish-owned contract manufacturer of electronics Elcoteq Hungary Kft has announced plans to recruit more than 650 new staffers to fulfill new orders in P+_cs , where the company has two plants .,
 The gross area of the Innova 2 project will be about 10,000 sq m

In [186]:
help(loaded_model.predict)

Help on method predict in module mlflow.pyfunc:

predict(data: Union[pandas.core.frame.DataFrame, pandas.core.series.Series, numpy.ndarray, ForwardRef('csc_matrix'), ForwardRef('csr_matrix'), List[Any], Dict[str, Any], datetime.datetime, bool, bytes, float, int, str], params: Optional[Dict[str, Any]] = None) -> Union[pandas.core.frame.DataFrame, pandas.core.series.Series, numpy.ndarray, list, str] method of mlflow.pyfunc.PyFuncModel instance
    Generates model predictions.
    
    If the model contains signature, enforce the input schema first before calling the model
    implementation with the sanitized input. If the pyfunc model does not include model schema,
    the input is passed to the model implementation as is. See `Model Signature Enforcement
    <https://www.mlflow.org/docs/latest/models.html#signature-enforcement>`_ for more details.
    
    Args:
        data: Model input as one of pandas.DataFrame, numpy.ndarray,
            scipy.sparse.(csc_matrix | csr_matrix), List

In [190]:
loaded_model.predict(data = test[["text"]])

,predictions
3207,"{'positive': 0.21659491956233978, 'negative': ..."
1684,"{'positive': 0.34155046939849854, 'negative': ..."
1044,"{'positive': 0.8569769263267517, 'negative': 0..."
4145,"{'positive': 0.255377858877182, 'negative': 0...."
1538,"{'positive': 0.24798519909381866, 'negative': ..."
...,...
3691,"{'positive': 0.2780744135379791, 'negative': 0..."
1507,"{'positive': 0.17686673998832703, 'negative': ..."
1126,"{'positive': 0.16717375814914703, 'negative': ..."
180,"{'positive': 0.9693070650100708, 'negative': 0..."


In [191]:
spacy_model(test_docs[0])

RuntimeError: [E896] There was an error using the static vectors. Ensure that the vectors of the vocab are properly initialized, or set 'include_static_vectors' to False.